In [1]:
import os
%pwd

'd:\\Data Science\\Codes\\02 MLOPs\\End-to-End-ML-Project-with-MLFlow\\research'

In [2]:
os.chdir("../")
%pwd 

'd:\\Data Science\\Codes\\02 MLOPs\\End-to-End-ML-Project-with-MLFlow'

In [3]:
from dataclasses import dataclass
from pathlib import Path 

@dataclass(frozen=True)
class ModelEvaluationConfig:

    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from mlProject.constants import * 
from mlProject.utils.common import read_yaml,create_directories,save_json

In [14]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.LogisticRegression
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri=os.getenv("MLFLOW_TRACKING_URI"),
           
        )

        return model_evaluation_config

In [15]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [16]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    
    def eval_metrics(self,actual, pred):
        accuracy = accuracy_score(actual, pred)
        precision = precision_score(actual, pred)
        recall = recall_score(actual, pred)
        return accuracy, precision, recall
    


    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (accuracy, precision, recall) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"accuracy": accuracy, "precision": precision, "recall": recall}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("precision", precision)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="LogisticRegressionModel")
            else:
                mlflow.sklearn.log_model(model, "model")

    


In [17]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-03-17 06:56:44,793: INFO: common: YAML file loaded successfully: config\config.yaml:]
[2026-03-17 06:56:44,800: INFO: common: YAML file loaded successfully: params.yaml:]
[2026-03-17 06:56:44,808: INFO: common: YAML file loaded successfully: schema.yaml:]
[2026-03-17 06:56:44,811: INFO: common: Directory created at: artifacts:]
[2026-03-17 06:56:44,813: INFO: common: Directory created at: artifacts/model_evaluation:]
[2026-03-17 06:56:48,048: INFO: common: JSON file saved at: artifacts\model_evaluation\metrics.json:]


2026/03/17 06:56:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 06:56:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LogisticRegressionModel'.
2026/03/17 06:57:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LogisticRegressionModel, version 1
Created version '1' of model 'LogisticRegressionModel'.


🏃 View run bemused-newt-918 at: https://dagshub.com/asadullahcreative/End-to-End-ML-Project-with-MLFlow.mlflow/#/experiments/0/runs/66318e8fcf4d429a97ee51141bba08dc
🧪 View experiment at: https://dagshub.com/asadullahcreative/End-to-End-ML-Project-with-MLFlow.mlflow/#/experiments/0
